# 대규모 모델 직접 배우기: 지식 편집
소개: 언어 모델의 편집 방법과 도구
> 언어 모델이 특정 지식을 기억하는 방식을 조작하고 싶나요? 적절한 편집 방법을 선택하여 특정 지식을 편집하고, 편집된 모델을 검증해 봅시다!

## 1. 이 튜토리얼의 목표:

- EasyEdit 라이브러리 사용법 익히기
- 언어 모델의 편집 방법 습득 (최소한의 방식으로)
- 다양한 유형의 편집 방법 선택 기준 및 적용 시나리오 이해

## 2. 사전 준비:
### 2.1 EasyEdit 소개

https://github.com/zjunlp/EasyEdit

EasyEdit은 GPT-J, Llama, GPT-NEO, GPT2, T5 등의 언어 모델을 편집하기 위한 Python 패키지입니다. 목표는 특정 지식에 대해 언어 모델의 동작을 효과적으로 변경하면서, 다른 입력에 대한 성능에 부정적인 영향을 주지 않는 것입니다. 또한 사용하기 쉽고 확장성이 뛰어납니다.

EasyEdit은 현재 널리 사용되는 편집 방법들을 통합하고 있습니다:
![](./assets/1.png)

### 2.2 주요 프레임워크

![](./assets/2.png)
EasyEdit은 통합된 Editor, Method, Evaluate 프레임워크를 포함하며, 각각 편집 시나리오, 편집 기술, 평가 방법을 나타냅니다.
- Editor: 작업 시나리오를 설명하며, 편집할 모델, 편집할 지식, 기타 필요한 하이퍼파라미터를 포함합니다.
- Method: 사용하는 구체적인 지식 편집 방법 (예: ROME, MEND 등).
- Evaluate: 지식 편집 성능을 평가하는 지표로, 신뢰성(Reliability), 일반성(Generality), 지역성(Locality), 이식성(Portability)을 포함합니다.
- Trainer: 일부 편집 방법은 훈련 과정이 필요하며, Trainer 모듈을 통해 구현됩니다.


## 3. 환경 설치:
```
git clone https://github.com/zjunlp/EasyEdit.git
(선택사항) conda create -n EasyEdit python=3.9.7
cd EasyEdit
pip install -r requirements.txt
```


## 4. 편집 예시
> 목표: GPT-2-XL의 지식 기억을 변경하여 메시(Lionel Messi)의 직업을 축구(football)에서 농구(basketball)로 바꿉니다.
절차:
- 편집 방법 선택, 파라미터 준비
- 지식 편집 데이터 준비
- Editor 인스턴스화
- 실행(Run)!

아래에서 ROME 방법을 예시로 구체적으로 설명합니다:
### 4.1 ROME
Jupiter Notebook: [https://colab.research.google.com/drive/1KkyWqyV3BjXCWfdrrgbR-QS3AAokVZbr?usp=sharing#scrollTo=zWfGkNb9FBJQ]
- 편집 방법 선택, 파라미터 준비
  - 편집 방법을 ROME으로 선택하고, ROME과 GPT2-XL에 필요한 파라미터를 준비합니다.
  - 예: alg_name: "ROME", model_name: "./hugging_cache/gpt2-xl" 또는 로컬 모델 경로, "device": 사용할 GPU 번호
  - 나머지 파라미터는 기본값 유지 가능
![](./assets/3.png)
- 지식 편집 데이터 준비
    ```
    prompts = ['Question:What sport does Lionel Messi play? Answer:'] # x_e
    ground_truth = ['football'] # y
    target_new = ['basketball'] # y_e
    subject = ['Lionel Messi'] 
    ```
- Editor 인스턴스화: 준비된 파라미터를 BaseEditor 클래스에 전달하여 인스턴스화하고 커스텀 Editor 인스턴스를 얻습니다.
    ```
    hparams = ROMEHyperParams.from_hparams('./hparams/ROME/gpt2-xl.yaml')
    editor=BaseEditor.from_hparams(hparams)
    ```
- 실행! edit 메서드 호출:
    ```
    metrics, edited_model, _ = editor.edit(
        prompts=prompts,
        ground_truth=ground_truth,
        target_new=target_new,
        subject=subject,
        keep_original_weight=False
    )
    ```
![](./assets/4.png)
> 참고: 처음 특정 모델을 편집할 때 Wikipedia 말뭉치를 다운로드하고, 해당 모델의 각 레이어 상태(stats_dir: "./data/stats")를 계산하여 저장합니다. 이후 편집 시에는 이를 재사용합니다. 따라서 첫 번째 편집은 시간이 걸릴 수 있으며, 네트워크 연결이 원활한 상태에서 인내심을 갖고 기다리세요.
### 4.2 검증 및 평가
editor.edit은 metrics(EasyEdit의 Evaluate 모듈에 의해 계산됨)를 반환합니다. 형식은:
![](./assets/5.png)
일반성, 지역성, 이식성 수치를 얻으려면 edit 메서드에 평가용 데이터를 전달해야 합니다.

지역성을 예시로 들면, edit 메서드가 지역성 지표를 계산하도록 하며, 즉 locality_inputs에서 모델 응답의 정확도를 측정합니다.
```
locality_inputs = {
    'neighborhood':{
        'prompt': ['Joseph Fischhof, the', 'Larry Bird is a professional', 'In Forssa, they understand'],
        'ground_truth': ['piano', 'basketball', 'Finnish']
    }
}
metrics, edited_model, _ = editor.edit(
    prompts=prompts,
    ground_truth=ground_truth,
    target_new=target_new,
    locality_inputs=locality_inputs,
    keep_original_weight=False
)
```
또는 편집 전후 모델의 생성 동작을 직접 비교할 수도 있습니다.
```
generation_prompts = [
    "Lionel Messi, the",
    "The law in Ikaalinen declares the language"
]

model = GPT2LMHeadModel.from_pretrained('./hugging_cache/gpt2').to('cuda')
batch = tokenizer(generation_prompts, return_tensors='pt', padding=True, max_length=30)

pre_edit_outputs = model.generate(
    input_ids=batch['input_ids'].to('cuda'),
    attention_mask=batch['attention_mask'].to('cuda'),
    max_new_tokens=3
)
post_edit_outputs = edited_model.generate(
    input_ids=batch['input_ids'].to('cuda'),
    attention_mask=batch['attention_mask'].to('cuda'),
    max_new_tokens=3
)
```


## 5. 대규모 편집 (선택사항)
### 5.1 배치 편집 (Batch edit)
여러 데이터를 병렬 리스트로 구성하여 edit 메서드에 동시에 전달하는 배치 편집이 가능하며, 이 경우 MEMIT이 최적의 방법입니다. (https://colab.research.google.com/drive/1P1lVklP8bTyh8uxxSuHnHwB91i-1LW6Z)
```
prompts = ['Question:What sport does Lionel Messi play? Answer:',
            'The law in Ikaalinen declares the language']
ground_truth = ['football', 'Finnish']
target_new = ['basketball', 'Swedish']
subject = ['Lionel Messi', 'Ikaalinen']
```
### 5.2 벤치마크에서 테스트
- Counterfact
- ZsRE
```
{
    "case_id": 4402,
    "pararel_idx": 11185,
    "requested_rewrite": {
      "prompt": "{} debuted on",
      "relation_id": "P449",
      "target_new": {
        "str": "CBS",
        "id": "Q43380"
      },
      "target_true": {
        "str": "MTV",
        "id": "Q43359"
      },
      "subject": "Singled Out"
    },
    "paraphrase_prompts": [
      "No one on the ground was injured.  v",
      "The sex ratio was 1063. Singled Out is to debut on"
    ],
    "neighborhood_prompts": [
      "Daria premieres on",
      "Teen Wolf was originally aired on",
      "Spider-Man: The New Animated Series was originally aired on",
      "Celebrity Deathmatch premiered on",
      "\u00c6on Flux premiered on",
      "My Super Psycho Sweet 16 premieres on",
      "Daria was released on",
      "Jersey Shore premiered on",
      "Skins was originally aired on",
      "All You've Got premiered on"
    ]
  }
  ```
https://github.com/zjunlp/EasyEdit/blob/main/examples/run_zsre_llama2.py 